In [14]:
import pandas as pd
import numpy as np

INPUT_PATH  = r"D:\bitirme\matchmaker_test\data\DrugCombinationData.tsv"
OUTPUT_PATH = r"D:\bitirme\matchmaker_test\data\DrugCombinationData_unique_triplets.tsv"

DRUG1_COL = "drug1_id"
DRUG2_COL = "drug2_id"
CELL_COL  = "cell_line"
SYNERGY_COL = "synergy_loewe"

ORDER_INVARIANT = True

# Load
df = pd.read_csv(INPUT_PATH, sep="\t")

# Ensure numeric
df[SYNERGY_COL] = pd.to_numeric(df[SYNERGY_COL], errors="coerce")

df = df.dropna(subset=[DRUG1_COL, DRUG2_COL, CELL_COL, SYNERGY_COL])

# Make drug pairs order-invariant
if ORDER_INVARIANT:
    d1 = df[DRUG1_COL].astype(str)
    d2 = df[DRUG2_COL].astype(str)
    df["_drugA"] = np.where(d1 <= d2, d1, d2)
    df["_drugB"] = np.where(d1 <= d2, d2, d1)
    group_cols = ["_drugA", "_drugB", CELL_COL]
else:
    group_cols = [DRUG1_COL, DRUG2_COL, CELL_COL]

# Mean aggregation (FULL precision)
out = (
    df.groupby(group_cols, as_index=False)[SYNERGY_COL]
      .mean()
)

# Rename back
if ORDER_INVARIANT:
    out = out.rename(columns={"_drugA": DRUG1_COL, "_drugB": DRUG2_COL})

# Reorder columns
out = out[[DRUG1_COL, DRUG2_COL, CELL_COL, SYNERGY_COL]]

# Sort by cell_line (as you requested)
out = out.sort_values(by=[CELL_COL, DRUG1_COL, DRUG2_COL], kind="mergesort").reset_index(drop=True)

# Save WITHOUT rounding
out.to_csv(OUTPUT_PATH, sep="\t", index=False)

print("Saved with full floating-point precision.")
print(out.head())

Saved with full floating-point precision.
                      drug1_id                     drug2_id cell_line  \
0  AAKJLRGGTJKAMG-UHFFFAOYSA-N  AHJRHEGDXFFMBM-UHFFFAOYSA-N    AsPC-1   
1  AAKJLRGGTJKAMG-UHFFFAOYSA-N  BCFGMOOMADDAQU-UHFFFAOYSA-N    AsPC-1   
2  AAKJLRGGTJKAMG-UHFFFAOYSA-N  BEUQXVWXFDOSAQ-UHFFFAOYSA-N    AsPC-1   
3  AAKJLRGGTJKAMG-UHFFFAOYSA-N  BKWJAKQVGHWELA-UHFFFAOYSA-N    AsPC-1   
4  AAKJLRGGTJKAMG-UHFFFAOYSA-N  DRMCATBEKSVAPL-UHFFFAOYSA-N    AsPC-1   

   synergy_loewe  
0           0.50  
1           0.00  
2           0.50  
3           0.25  
4           0.50  


In [19]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator
from rdkit import DataStructs

SMILES_PATH = r"D:\bitirme\matchmaker_test\drug_smiles_with_ids.csv"
OUT_ALL_DRUG_FEATS = r"D:\bitirme\matchmaker_test\data\drug_chem_rdkit.csv"

FP_BITS = 2048
FP_RADIUS = 2  # radius=2 ~ ECFP4

# Load smiles
sm = pd.read_csv(SMILES_PATH)
sm.columns = [c.strip() for c in sm.columns]
sm = sm.loc[:, ~sm.columns.duplicated()]  # drop duplicate header names if any

required = ["drug_id", "smiles"]
missing = [c for c in required if c not in sm.columns]
if missing:
    raise ValueError(f"Missing columns in SMILES file: {missing}. Found: {list(sm.columns)}")

sm["drug_id"] = sm["drug_id"].astype(str)
sm["smiles"] = sm["smiles"].astype(str)
sm = sm.drop_duplicates(subset=["drug_id"], keep="first").reset_index(drop=True)

# New generator API (no warnings)
gen = GetMorganGenerator(radius=FP_RADIUS, fpSize=FP_BITS)

def smiles_to_bits(smi: str):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    fp = gen.GetFingerprint(mol)  # ExplicitBitVect
    arr = np.zeros((FP_BITS,), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

bad = []
rows = []
for _, r in sm.iterrows():
    vec = smiles_to_bits(r["smiles"])
    if vec is None:
        bad.append(r["drug_id"])
        continue
    rows.append([r["drug_id"], *vec.tolist()])

cols = ["drug_id"] + [f"fp_{i}" for i in range(FP_BITS)]
feat_df = pd.DataFrame(rows, columns=cols)
feat_df.to_csv(OUT_ALL_DRUG_FEATS, index=False)

print(f"Saved drug features: {OUT_ALL_DRUG_FEATS}")
print(f"Drugs in SMILES file: {len(sm):,}")
print(f"Drugs with valid RDKit mol: {len(feat_df):,}")
if bad:
    print(f"Invalid SMILES for {len(bad)} drugs (example 10): {bad[:10]}")

Saved drug features: D:\bitirme\matchmaker_test\data\drug_chem_rdkit.csv
Drugs in SMILES file: 57
Drugs with valid RDKit mol: 57


In [20]:
feat_df.iloc[:, 1:] = feat_df.iloc[:, 1:].astype(np.float32)

C:\Users\ardat\AppData\Local\Temp\ipykernel_15512\1691217139.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '0     0.0
1     0.0
2     0.0
3     0.0
4     0.0
5     0.0
6     0.0
7     0.0
8     0.0
9     0.0
10    0.0
11    0.0
12    0.0
13    0.0
14    0.0
15    0.0
16    0.0
17    0.0
18    0.0
19    0.0
20    0.0
21    0.0
22    0.0
23    0.0
24    0.0
25    0.0
26    0.0
27    0.0
28    0.0
29    0.0
30    0.0
31    0.0
32    0.0
33    0.0
34    0.0
35    0.0
36    0.0
37    0.0
38    0.0
39    0.0
40    0.0
41    0.0
42    0.0
43    0.0
44    0.0
45    0.0
46    0.0
47    0.0
48    0.0
49    0.0
50    0.0
51    0.0
52    0.0
53    0.0
54    0.0
55    0.0
56    0.0
Name: fp_0, dtype: float32' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  feat_df.iloc[:, 1:] = feat_df.iloc[:, 1:].astype(np.float32)
C:\Users\ardat\AppData\Local\Temp\ipykernel_15512\1691217139.py:1: 

In [21]:
import pandas as pd

COMBO_PATH = r"D:\bitirme\matchmaker_test\data\DrugCombinationData_unique_triplets.tsv"
DRUG_FEATS_PATH = r"D:\bitirme\matchmaker_test\data\drug_chem_rdkit.csv"

OUT_DRUG1 = r"D:\bitirme\matchmaker_test\data\drug1_chem_new.csv"
OUT_DRUG2 = r"D:\bitirme\matchmaker_test\data\drug2_chem_new.csv"

combo = pd.read_csv(COMBO_PATH, sep="\t")
drug_feats = pd.read_csv(DRUG_FEATS_PATH)

drug_feats["drug_id"] = drug_feats["drug_id"].astype(str)

drug1_ids = set(combo["drug1_id"].astype(str).unique())
drug2_ids = set(combo["drug2_id"].astype(str).unique())

drug1_df = drug_feats[drug_feats["drug_id"].isin(drug1_ids)].copy()
drug2_df = drug_feats[drug_feats["drug_id"].isin(drug2_ids)].copy()

# Rename id column to match your pipeline expectation (optional)
drug1_df = drug1_df.rename(columns={"drug_id": "drug1_id"})
drug2_df = drug2_df.rename(columns={"drug_id": "drug2_id"})

drug1_df.to_csv(OUT_DRUG1, index=False)
drug2_df.to_csv(OUT_DRUG2, index=False)

print(f"Saved: {OUT_DRUG1} ({len(drug1_df):,} rows)")
print(f"Saved: {OUT_DRUG2} ({len(drug2_df):,} rows)")

# Quick coverage check
missing1 = drug1_ids - set(drug1_df["drug1_id"])
missing2 = drug2_ids - set(drug2_df["drug2_id"])
print("Missing drug1 features:", len(missing1))
print("Missing drug2 features:", len(missing2))
if missing1:
    print("Example missing drug1:", list(missing1)[:10])
if missing2:
    print("Example missing drug2:", list(missing2)[:10])

Saved: D:\bitirme\matchmaker_test\data\drug1_chem_new.csv (25 rows)
Saved: D:\bitirme\matchmaker_test\data\drug2_chem_new.csv (25 rows)
Missing drug1 features: 0
Missing drug2 features: 0


In [22]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

# ---------------- PATHS ----------------
COMBO_PATH  = r"D:\bitirme\matchmaker_test\data\DrugCombinationData_unique_triplets.tsv"
SMILES_PATH = r"D:\bitirme\matchmaker_test\drug_smiles_with_ids.csv"

OUT_MATRIX = r"D:\bitirme\matchmaker_test\data\drug_chem_rdkit_noheader.csv"
OUT_ORDER  = r"D:\bitirme\matchmaker_test\data\drug_ids_order.txt"

# ---------------- COLUMN NAMES ----------------
DRUG1_COL = "drug1_id"
DRUG2_COL = "drug2_id"
SMILES_ID_COL = "drug_id"
SMILES_COL = "smiles"

# ---------------- 1) Load combo + decide drug order ----------------
combo = pd.read_csv(COMBO_PATH, sep="\t")

# Keep the order "as it appears" in the file (stable)
ordered_drugs = pd.unique(pd.concat([combo[DRUG1_COL].astype(str), combo[DRUG2_COL].astype(str)], ignore_index=True))

# ---------------- 2) Load smiles ----------------
sm = pd.read_csv(SMILES_PATH)
sm.columns = [c.strip() for c in sm.columns]
sm = sm.loc[:, ~sm.columns.duplicated()]  # if your csv had duplicated headers

sm[SMILES_ID_COL] = sm[SMILES_ID_COL].astype(str)
sm[SMILES_COL] = sm[SMILES_COL].astype(str)

id2smi = dict(zip(sm[SMILES_ID_COL], sm[SMILES_COL]))

# ---------------- 3) Build RDKit descriptor calculator ----------------
# Use RDKit’s built-in descriptor list (continuous floats)
desc_names = [name for name, func in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)

def featurize_smiles(smi: str):
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None
    vals = np.array(calc.CalcDescriptors(mol), dtype=np.float64)
    # Replace inf/-inf with nan for safety
    vals[~np.isfinite(vals)] = np.nan
    return vals

# ---------------- 4) Featurize in the chosen order ----------------
features = []
missing = []
bad_smiles = []

for did in ordered_drugs:
    smi = id2smi.get(did, None)
    if smi is None:
        missing.append(did)
        continue
    vec = featurize_smiles(smi)
    if vec is None:
        bad_smiles.append(did)
        continue
    features.append(vec)

# Check consistency
if len(features) == 0:
    raise RuntimeError("No features generated. Check column names and SMILES validity.")

X = np.vstack(features)

# Optional: impute NaNs with column means (common for descriptors)
col_means = np.nanmean(X, axis=0)
inds = np.where(np.isnan(X))
X[inds] = np.take(col_means, inds[1])

# ---------------- 5) Save: NO HEADER numeric matrix ----------------
# fmt="%.15g" avoids forced trailing zeros like 2.500000000
np.savetxt(OUT_MATRIX, X, delimiter=",", fmt="%.15g")

# Save the row order mapping (VERY important!)
with open(OUT_ORDER, "w", encoding="utf-8") as f:
    for did in ordered_drugs:
        f.write(did + "\n")

print("Saved matrix (no header):", OUT_MATRIX)
print("Saved drug order mapping:", OUT_ORDER)
print("Matrix shape:", X.shape)

if missing:
    print("Missing SMILES for drugs:", len(missing), "example:", missing[:10])
if bad_smiles:
    print("Bad/invalid SMILES:", len(bad_smiles), "example:", bad_smiles[:10])

Saved matrix (no header): D:\bitirme\matchmaker_test\data\drug_chem_rdkit_noheader.csv
Saved drug order mapping: D:\bitirme\matchmaker_test\data\drug_ids_order.txt
Matrix shape: (26, 217)


In [24]:
import pandas as pd
import numpy as np

from rdkit import Chem
from rdkit.Chem import Descriptors
from rdkit.ML.Descriptors import MoleculeDescriptors

# ---------------- PATHS ----------------
COMBO_PATH  = r"D:\bitirme\matchmaker_test\data\DrugCombinationData_unique_triplets.tsv"
SMILES_PATH = r"D:\bitirme\matchmaker_test\drug_smiles_with_ids.csv"

OUT_DRUG1_CHEM = r"D:\bitirme\matchmaker_test\data\drug1_chem_new.csv"
OUT_DRUG2_CHEM = r"D:\bitirme\matchmaker_test\data\drug2_chem_new.csv"
OUT_DRUG1_ORDER = r"D:\bitirme\matchmaker_test\data\drug1_ids_order.txt"
OUT_DRUG2_ORDER = r"D:\bitirme\matchmaker_test\data\drug2_ids_order.txt"

# ---------------- COLUMN NAMES ----------------
DRUG1_COL = "drug1_id"
DRUG2_COL = "drug2_id"
SMILES_ID_COL = "drug_id"
SMILES_COL = "smiles"

# ---------------- Load combo ----------------
combo = pd.read_csv(COMBO_PATH, sep="\t")
combo[DRUG1_COL] = combo[DRUG1_COL].astype(str)
combo[DRUG2_COL] = combo[DRUG2_COL].astype(str)

# Keep "first seen" order (matches your file order)
drug1_order = pd.unique(combo[DRUG1_COL])
drug2_order = pd.unique(combo[DRUG2_COL])

# ---------------- Load smiles ----------------
sm = pd.read_csv(SMILES_PATH)
sm.columns = [c.strip() for c in sm.columns]
sm = sm.loc[:, ~sm.columns.duplicated()]  # in case duplicate headers exist
sm[SMILES_ID_COL] = sm[SMILES_ID_COL].astype(str)
sm[SMILES_COL] = sm[SMILES_COL].astype(str)

id2smi = dict(zip(sm[SMILES_ID_COL], sm[SMILES_COL]))

# ---------------- RDKit descriptor calculator ----------------
desc_names = [name for name, _ in Descriptors._descList]
calc = MoleculeDescriptors.MolecularDescriptorCalculator(desc_names)

def featurize(drug_id: str):
    smi = id2smi.get(drug_id, None)
    if smi is None:
        return None, "missing_smiles"
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        return None, "bad_smiles"
    v = np.array(calc.CalcDescriptors(mol), dtype=np.float64)
    v[~np.isfinite(v)] = np.nan
    return v, None

def build_matrix(drug_order):
    rows = []
    missing = []
    bad = []
    for did in drug_order:
        vec, err = featurize(did)
        if err == "missing_smiles":
            missing.append(did)
            continue
        if err == "bad_smiles":
            bad.append(did)
            continue
        rows.append(vec)

    X = np.vstack(rows) if rows else np.empty((0, len(desc_names)))
    # Impute NaNs with column means
    if X.size > 0:
        col_means = np.nanmean(X, axis=0)
        inds = np.where(np.isnan(X))
        X[inds] = np.take(col_means, inds[1])

    return X, missing, bad

# ---------------- Build drug1/drug2 matrices ----------------
X1, miss1, bad1 = build_matrix(drug1_order)
X2, miss2, bad2 = build_matrix(drug2_order)

# ---------------- Save (NO HEADER) ----------------
# fmt="%.15g" avoids ugly trailing zeros (2.500000000) but keeps precision
np.savetxt(OUT_DRUG1_CHEM, X1, delimiter=",", fmt="%.15g")
np.savetxt(OUT_DRUG2_CHEM, X2, delimiter=",", fmt="%.15g")

# Save row order mapping
with open(OUT_DRUG1_ORDER, "w", encoding="utf-8") as f:
    for did in drug1_order:
        f.write(did + "\n")
with open(OUT_DRUG2_ORDER, "w", encoding="utf-8") as f:
    for did in drug2_order:
        f.write(did + "\n")

print("Saved:", OUT_DRUG1_CHEM, "shape:", X1.shape)
print("Saved:", OUT_DRUG2_CHEM, "shape:", X2.shape)
print("Saved:", OUT_DRUG1_ORDER)
print("Saved:", OUT_DRUG2_ORDER)

if miss1 or bad1 or miss2 or bad2:
    print("\n--- Issues ---")
    if miss1: print("drug1 missing smiles:", len(miss1), "example:", miss1[:10])
    if bad1:  print("drug1 bad smiles:", len(bad1), "example:", bad1[:10])
    if miss2: print("drug2 missing smiles:", len(miss2), "example:", miss2[:10])
    if bad2:  print("drug2 bad smiles:", len(bad2), "example:", bad2[:10])

Saved: D:\bitirme\matchmaker_test\data\drug1_chem_new.csv shape: (25, 217)
Saved: D:\bitirme\matchmaker_test\data\drug2_chem_new.csv shape: (25, 217)
Saved: D:\bitirme\matchmaker_test\data\drug1_ids_order.txt
Saved: D:\bitirme\matchmaker_test\data\drug2_ids_order.txt


In [26]:
import numpy as np
import pandas as pd

# ---------- inputs ----------
COMBO_PATH = r"D:\bitirme\matchmaker_test\data\DrugCombinationData_unique_triplets.tsv"

DRUG1_CHEM = r"D:\bitirme\matchmaker_test\data\drug1_chem_new.csv"
DRUG2_CHEM = r"D:\bitirme\matchmaker_test\data\drug2_chem_new.csv"
DRUG1_ORDER = r"D:\bitirme\matchmaker_test\data\drug1_ids_order.txt"
DRUG2_ORDER = r"D:\bitirme\matchmaker_test\data\drug2_ids_order.txt"

# ---------- outputs ----------
OUT_DRUG1_EXP = r"D:\bitirme\matchmaker_test\data\drug1_chem_expanded.csv"
OUT_DRUG2_EXP = r"D:\bitirme\matchmaker_test\data\drug2_chem_expanded.csv"

# ---------- columns ----------
DRUG1_COL = "drug1_id"
DRUG2_COL = "drug2_id"

# ---------- load combo ----------
combo = pd.read_csv(COMBO_PATH, sep="\t")
combo[DRUG1_COL] = combo[DRUG1_COL].astype(str)
combo[DRUG2_COL] = combo[DRUG2_COL].astype(str)

# ---------- load chem matrices (no header) + orders ----------
X1 = np.loadtxt(DRUG1_CHEM, delimiter=",")
X2 = np.loadtxt(DRUG2_CHEM, delimiter=",")

with open(DRUG1_ORDER, "r", encoding="utf-8") as f:
    drug1_ids = [line.strip() for line in f if line.strip()]
with open(DRUG2_ORDER, "r", encoding="utf-8") as f:
    drug2_ids = [line.strip() for line in f if line.strip()]

if X1.shape[0] != len(drug1_ids):
    raise ValueError(f"drug1_chem rows ({X1.shape[0]}) != drug1_ids_order length ({len(drug1_ids)})")
if X2.shape[0] != len(drug2_ids):
    raise ValueError(f"drug2_chem rows ({X2.shape[0]}) != drug2_ids_order length ({len(drug2_ids)})")

drug1_index = {did: i for i, did in enumerate(drug1_ids)}
drug2_index = {did: i for i, did in enumerate(drug2_ids)}

# ---------- expand to match combo rows ----------
idx1 = combo[DRUG1_COL].map(drug1_index).to_numpy()
idx2 = combo[DRUG2_COL].map(drug2_index).to_numpy()

if np.any(pd.isna(idx1)) or np.any(pd.isna(idx2)):
    missing1 = combo.loc[pd.isna(idx1), DRUG1_COL].unique().tolist()
    missing2 = combo.loc[pd.isna(idx2), DRUG2_COL].unique().tolist()
    raise ValueError(
        f"Missing drug features.\n"
        f"Missing in drug1_index: {missing1[:10]} (total {len(missing1)})\n"
        f"Missing in drug2_index: {missing2[:10]} (total {len(missing2)})"
    )

idx1 = idx1.astype(int)
idx2 = idx2.astype(int)

X1_exp = X1[idx1, :]
X2_exp = X2[idx2, :]

# Save headerless; fmt avoids ugly trailing zeros but keeps precision
np.savetxt(OUT_DRUG1_EXP, X1_exp, delimiter=",", fmt="%.15g")
np.savetxt(OUT_DRUG2_EXP, X2_exp, delimiter=",", fmt="%.15g")

print("drug1 expanded:", X1_exp.shape, "->", OUT_DRUG1_EXP)
print("drug2 expanded:", X2_exp.shape, "->", OUT_DRUG2_EXP)
print("combo rows:", len(combo))

drug1 expanded: (9397, 217) -> D:\bitirme\matchmaker_test\data\drug1_chem_expanded.csv
drug2 expanded: (9397, 217) -> D:\bitirme\matchmaker_test\data\drug2_chem_expanded.csv
combo rows: 9397


In [29]:
import numpy as np
import pandas as pd

# ---------- inputs ----------
COMBO_PATH = r"D:\bitirme\matchmaker_test\data\DrugCombinationData_unique_triplets.tsv"
RNASEQ_PATH = r"D:\bitirme\matchmaker_yeni\rnaseq_pdac_subset.csv"
LANDMARK_PATH = r"D:\bitirme\matchmaker_arda\landmark_genes_972.txt"

# ---------- outputs ----------
OUT_CELL_LINE_GEX = r"D:\bitirme\matchmaker_test\data\cell_line_gex_new.csv"   # cell_line x genes, headerless
OUT_CELL_LINE_ORDER = r"D:\bitirme\matchmaker_test\data\cell_line_order.txt"  # row->cell_line mapping
OUT_GENE_ORDER = r"D:\bitirme\matchmaker_test\data\gex_gene_order.txt"        # col->gene mapping

# ---------- column names ----------
COMBO_CELL_COL = "cell_line"     # change to "cell_line_name" if needed
RNASEQ_CELL_COL = "model_name"
GENE_COL = "gene_symbol"
VAL_COL = "rsem_tpm"             # you can change to rsem_fpkm or htseq_fpkm

# ---------- load combo ----------
combo = pd.read_csv(COMBO_PATH, sep="\t")
combo.columns = [c.strip() for c in combo.columns]

if COMBO_CELL_COL not in combo.columns:
    raise ValueError(f"Combo file missing '{COMBO_CELL_COL}'. Columns: {list(combo.columns)}")

combo[COMBO_CELL_COL] = combo[COMBO_CELL_COL].astype(str)
cell_order = pd.unique(combo[COMBO_CELL_COL])  # stable order as appears in combo

# ---------- load landmark genes ----------
with open(LANDMARK_PATH, "r", encoding="utf-8") as f:
    genes = [line.strip() for line in f if line.strip()]
gene_set = set(genes)

# ---------- load rnaseq (long format) ----------
rna = pd.read_csv(RNASEQ_PATH)
rna.columns = [c.strip() for c in rna.columns]

needed = [RNASEQ_CELL_COL, GENE_COL, VAL_COL]
missing = [c for c in needed if c not in rna.columns]
if missing:
    raise ValueError(f"rnaseq file missing columns {missing}. Columns: {list(rna.columns)}")

rna[RNASEQ_CELL_COL] = rna[RNASEQ_CELL_COL].astype(str)
rna[GENE_COL] = rna[GENE_COL].astype(str)
rna[VAL_COL] = pd.to_numeric(rna[VAL_COL], errors="coerce")

# Filter to cell lines in combo + landmark genes
rna_f = rna[rna[RNASEQ_CELL_COL].isin(cell_order) & rna[GENE_COL].isin(gene_set)].copy()

# Pivot to cell_line x gene
piv = rna_f.pivot_table(
    index=RNASEQ_CELL_COL,
    columns=GENE_COL,
    values=VAL_COL,
    aggfunc="mean"
)

# Reindex rows/cols to desired order
piv = piv.reindex(index=cell_order)
piv = piv.reindex(columns=genes)

mat = piv.to_numpy(dtype=np.float64)

# Impute NaNs with column means (handles missing genes/cell lines)
col_means = np.nanmean(mat, axis=0)
inds = np.where(np.isnan(mat))
mat[inds] = np.take(col_means, inds[1])

# Save headerless matrix
np.savetxt(OUT_CELL_LINE_GEX, mat, delimiter=",", fmt="%.15g")

# Save mapping files
with open(OUT_CELL_LINE_ORDER, "w", encoding="utf-8") as f:
    for c in piv.index.tolist():
        f.write(str(c) + "\n")

with open(OUT_GENE_ORDER, "w", encoding="utf-8") as f:
    for g in piv.columns.tolist():
        f.write(str(g) + "\n")

print("Saved cell_line_gex_new:", mat.shape, "->", OUT_CELL_LINE_GEX)
print("Saved cell_line_order  :", OUT_CELL_LINE_ORDER)
print("Saved gene_order       :", OUT_GENE_ORDER)

# Coverage diagnostics
missing_cells = [c for c in cell_order if c not in set(rna[RNASEQ_CELL_COL])]
present_cells = len(cell_order) - len(missing_cells)
print(f"Combo cell lines: {len(cell_order)}, present in rnaseq: {present_cells}, missing: {len(missing_cells)}")
if missing_cells:
    print("Example missing cell lines:", missing_cells[:10])

C:\Users\ardat\AppData\Local\Temp\ipykernel_15512\2184808360.py:66: RuntimeWarning: Mean of empty slice
  col_means = np.nanmean(mat, axis=0)


Saved cell_line_gex_new: (29, 972) -> D:\bitirme\matchmaker_test\data\cell_line_gex_new.csv
Saved cell_line_order  : D:\bitirme\matchmaker_test\data\cell_line_order.txt
Saved gene_order       : D:\bitirme\matchmaker_test\data\gex_gene_order.txt
Combo cell lines: 29, present in rnaseq: 29, missing: 0


In [30]:
import numpy as np
import pandas as pd

# ---------- inputs ----------
COMBO_PATH = r"D:\bitirme\matchmaker_test\data\DrugCombinationData_unique_triplets.tsv"

GEX_UNIQUE = r"D:\bitirme\matchmaker_test\data\cell_line_gex_new.csv"   # unique cell_line x genes
CELL_ORDER_PATH = r"D:\bitirme\matchmaker_test\data\cell_line_order.txt"

# ---------- outputs ----------
GEX_EXPANDED = r"D:\bitirme\matchmaker_test\data\cell_line_gex_expanded.csv"

# Optional: also output filtered versions with zero-only cell lines removed
DROP_ZERO_CELL_LINES = True
OUT_COMBO_FILTERED = r"D:\bitirme\matchmaker_test\data\DrugCombinationData_unique_triplets_nozero.tsv"
OUT_GEX_EXPANDED_FILTERED = r"D:\bitirme\matchmaker_test\data\cell_line_gex_expanded_nozero.csv"

# ---------- combo cell line column ----------
COMBO_CELL_COL = "cell_line"  # change if your combo uses cell_line_name

# ---------- load combo ----------
combo = pd.read_csv(COMBO_PATH, sep="\t")
combo.columns = [c.strip() for c in combo.columns]
combo[COMBO_CELL_COL] = combo[COMBO_CELL_COL].astype(str)

# ---------- load unique gex ----------
G = np.loadtxt(GEX_UNIQUE, delimiter=",")  # shape: (#cell_lines, #genes)

with open(CELL_ORDER_PATH, "r", encoding="utf-8") as f:
    cell_lines = [line.strip() for line in f if line.strip()]

if G.shape[0] != len(cell_lines):
    raise ValueError(f"GEX rows ({G.shape[0]}) != cell_line_order length ({len(cell_lines)})")

cell_index = {c: i for i, c in enumerate(cell_lines)}

# ---------- detect zero rows in unique matrix ----------
row_sums = G.sum(axis=1)
zero_rows_idx = np.where(row_sums == 0)[0]
zero_cells = [cell_lines[i] for i in zero_rows_idx]

print("Unique GEX shape:", G.shape)
print("Zero-only cell lines in unique GEX:", len(zero_cells))
if zero_cells:
    print("Zero-only cells:", zero_cells)

# ---------- expand ----------
idx = combo[COMBO_CELL_COL].map(cell_index)

if idx.isna().any():
    missing = combo.loc[idx.isna(), COMBO_CELL_COL].unique().tolist()
    raise ValueError(
        f"Some combo cell lines are missing in cell_line_order.txt: {missing[:20]} "
        f"(total {len(missing)})"
    )

idx = idx.astype(int).to_numpy()
G_exp = G[idx, :]

np.savetxt(GEX_EXPANDED, G_exp, delimiter=",", fmt="%.15g")
print("Expanded GEX shape:", G_exp.shape, "->", GEX_EXPANDED)
print("Combo rows:", len(combo))

# ---------- optional: drop samples where their cell line is zero-only ----------
if DROP_ZERO_CELL_LINES and zero_cells:
    mask_keep = ~combo[COMBO_CELL_COL].isin(zero_cells)
    combo_f = combo.loc[mask_keep].reset_index(drop=True)
    G_exp_f = G_exp[mask_keep.to_numpy(), :]

    combo_f.to_csv(OUT_COMBO_FILTERED, sep="\t", index=False)
    np.savetxt(OUT_GEX_EXPANDED_FILTERED, G_exp_f, delimiter=",", fmt="%.15g")

    print("\nDropped zero-only cell line samples.")
    print("Filtered combo rows:", len(combo_f), "->", OUT_COMBO_FILTERED)
    print("Filtered expanded GEX:", G_exp_f.shape, "->", OUT_GEX_EXPANDED_FILTERED)

Unique GEX shape: (29, 972)
Zero-only cell lines in unique GEX: 0
Expanded GEX shape: (9397, 972) -> D:\bitirme\matchmaker_test\data\cell_line_gex_expanded.csv
Combo rows: 9397


In [33]:
import numpy as np
import pandas as pd

COMBO_PATH = r"data/DrugCombinationData.tsv"  # or your full path
df = pd.read_csv(COMBO_PATH, sep="\t")
N = len(df)

rng = np.random.default_rng(42)
idx = np.arange(N)
rng.shuffle(idx)

train_end = int(0.8 * N)
val_end   = int(0.9 * N)

train_inds = idx[:train_end]
val_inds   = idx[train_end:val_end]
test_inds  = idx[val_end:]

np.savetxt("data/train_inds.txt", train_inds, fmt="%d")
np.savetxt("data/val_inds.txt",   val_inds,   fmt="%d")
np.savetxt("data/test_inds.txt",  test_inds,  fmt="%d")

print("Wrote new splits for N =", N)
print("Train:", len(train_inds), "min/max:", train_inds.min(), train_inds.max())
print("Val  :", len(val_inds),   "min/max:", val_inds.min(),   val_inds.max())
print("Test :", len(test_inds),  "min/max:", test_inds.min(),  test_inds.max())

Wrote new splits for N = 9397
Train: 7517 min/max: 1 9396
Val  : 940 min/max: 0 9395
Test : 940 min/max: 5 9375


In [34]:
import numpy as np
for f in ["data/train_inds.txt","data/val_inds.txt","data/test_inds.txt"]:
    x = np.loadtxt(f, dtype=int)
    print(f, "min", x.min(), "max", x.max(), "count", len(x))

data/train_inds.txt min 1 max 9396 count 7517
data/val_inds.txt min 0 max 9395 count 940
data/test_inds.txt min 5 max 9375 count 940


In [35]:
import numpy as np
import pandas as pd

combo = pd.read_csv("data/DrugCombinationData.tsv", sep="\t")
N = len(combo)

train = np.loadtxt("data/train_inds.txt", dtype=int)
val   = np.loadtxt("data/val_inds.txt", dtype=int)
test  = np.loadtxt("data/test_inds.txt", dtype=int)

print("Total samples:", N)
print("Train:", len(train), f"({len(train)/N:.3f})")
print("Val  :", len(val),   f"({len(val)/N:.3f})")
print("Test :", len(test),  f"({len(test)/N:.3f})")

Total samples: 9397
Train: 7517 (0.800)
Val  : 940 (0.100)
Test : 940 (0.100)
